# YOLO 라벨 재생성 + 재학습 (FC코드 71종 → 메뉴명 N종)

Drive `finetune_dataset/`의 이미지는 그대로, 라벨 txt만 재생성.
파일명에서 메뉴명 추출 → 새 class_id 부여 → data.yaml 재생성 → 재학습.

**소요 시간:** 라벨 재생성 ~5분 + 학습 A100 기준 ~1시간

In [ ]:
# Step 1: Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: 경로 확인 + 현황 파악
import os
from pathlib import Path

DATASET = Path('/content/drive/MyDrive/LGHellovision/Project 02/Object Detection/finetune_dataset')

splits = ['train', 'val']
for split in splits:
    img_dir = DATASET / split / 'images'
    lbl_dir = DATASET / split / 'labels'
    if img_dir.exists():
        img_count = len([f for f in os.listdir(str(img_dir)) if f.endswith('.jpg')])
    else:
        # images가 바로 아래 있을 수도
        img_dir = DATASET / 'images' / split
        lbl_dir = DATASET / 'labels' / split
        img_count = len([f for f in os.listdir(str(img_dir)) if f.endswith('.jpg')]) if img_dir.exists() else 0
    lbl_count = len([f for f in os.listdir(str(lbl_dir)) if f.endswith('.txt')]) if lbl_dir.exists() else 0
    print(f'{split}: images={img_count}, labels={lbl_count}')
    print(f'  img_dir: {img_dir}')
    print(f'  lbl_dir: {lbl_dir}')

print('\n위 경로가 맞는지 확인하세요. 틀리면 아래 SPLITS 딕셔너리를 수정.')

In [ ]:
# Step 3: 이미지 파일명에서 메뉴명 추출 + 새 클래스 매핑
#
# 파일명: A_13_A13001_가자미구이_30_10.jpg
# parts:  [A, 13, A13001, 가자미구이, 30, 10]
#                          ↑ 메뉴명 (parts[3])

from collections import defaultdict

DATASET = Path('/content/drive/MyDrive/LGHellovision/Project 02/Object Detection/finetune_dataset')

# ★ Step 2 출력 보고 실제 경로로 수정
SPLITS = {
    'train': {
        'img': DATASET / 'train' / 'images',
        'lbl': DATASET / 'train' / 'labels',
    },
    'val': {
        'img': DATASET / 'val' / 'images',
        'lbl': DATASET / 'val' / 'labels',
    },
}
# images/train, labels/train 구조면 아래로 교체
# SPLITS = {
#     'train': {'img': DATASET / 'images' / 'train', 'lbl': DATASET / 'labels' / 'train'},
#     'val':   {'img': DATASET / 'images' / 'val',   'lbl': DATASET / 'labels' / 'val'},
# }

# 전체 이미지 스캔 → 메뉴명 수집
menu_count = defaultdict(int)
parse_errors = []

for split, paths in SPLITS.items():
    img_dir = paths['img']
    if not img_dir.exists():
        print(f'[WARN] {img_dir} 없음')
        continue
    for fname in os.listdir(str(img_dir)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        parts = fname.rsplit('.', 1)[0].split('_')
        if len(parts) < 4:
            parse_errors.append(fname)
            continue
        menu_name = parts[3]
        menu_count[menu_name] += 1

# 빈도순 정렬 → class_id 부여
sorted_menus = sorted(menu_count.items(), key=lambda x: -x[1])
menu_to_id = {name: i for i, (name, _) in enumerate(sorted_menus)}

print(f'고유 메뉴 수: {len(menu_to_id)}')
print(f'총 이미지 수: {sum(menu_count.values())}')
print(f'파싱 실패: {len(parse_errors)}건')
print(f'\n이미지 수 상위 20개 메뉴:')
for name, cnt in sorted_menus[:20]:
    print(f'  [{menu_to_id[name]:>3}] {name:<20} {cnt:>5}장')
print(f'\n이미지 수 하위 10개 메뉴:')
for name, cnt in sorted_menus[-10:]:
    print(f'  [{menu_to_id[name]:>3}] {name:<20} {cnt:>5}장')

In [ ]:
# Step 4: 라벨 txt 재생성
#
# 기존 라벨: "old_class_id cx cy w h"
# 새 라벨:   "new_class_id cx cy w h"  (bbox 좌표 그대로, class_id만 교체)
#
# 이미지 파일명에서 메뉴명 → menu_to_id → 새 class_id

import shutil

total_updated = 0
total_skipped = 0

for split, paths in SPLITS.items():
    img_dir = paths['img']
    lbl_dir = paths['lbl']
    if not lbl_dir.exists():
        print(f'[WARN] {lbl_dir} 없음 — 스킵')
        continue

    # 백업
    backup_dir = lbl_dir.parent / f'{lbl_dir.name}_backup_fc71'
    if not backup_dir.exists():
        print(f'[{split}] 기존 라벨 백업: {backup_dir}')
        shutil.copytree(str(lbl_dir), str(backup_dir))
    else:
        print(f'[{split}] 백업 이미 존재: {backup_dir}')

    updated = 0
    skipped = 0
    label_files = [f for f in os.listdir(str(lbl_dir)) if f.endswith('.txt')]

    for i, lbl_file in enumerate(label_files):
        if i % 1000 == 0 and i > 0:
            print(f'  [{split}] {i:,}/{len(label_files):,} 처리 중...', end='\r')

        stem = lbl_file.rsplit('.', 1)[0]
        parts = stem.split('_')
        if len(parts) < 4:
            skipped += 1
            continue

        menu_name = parts[3]
        if menu_name not in menu_to_id:
            skipped += 1
            continue

        new_id = menu_to_id[menu_name]
        lbl_path = lbl_dir / lbl_file

        # 기존 라벨 읽기
        with open(lbl_path, 'r') as f:
            lines = f.readlines()

        # class_id만 교체
        new_lines = []
        for line in lines:
            tokens = line.strip().split()
            if len(tokens) >= 5:
                tokens[0] = str(new_id)
                new_lines.append(' '.join(tokens) + '\n')

        if new_lines:
            with open(lbl_path, 'w') as f:
                f.writelines(new_lines)
            updated += 1
        else:
            skipped += 1

    print(f'  [{split}] 완료 — 업데이트: {updated:,} / 스킵: {skipped:,}')
    total_updated += updated
    total_skipped += skipped

print(f'\n전체: {total_updated:,}개 라벨 업데이트, {total_skipped:,}개 스킵')

In [ ]:
# Step 5: data.yaml 재생성
import yaml

names_list = [name for name, _ in sorted_menus]

data_yaml = {
    'train': './train/images',
    'val': './val/images',
    'nc': len(names_list),
    'names': names_list,
}

yaml_path = DATASET / 'data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False)

print(f'data.yaml 저장: {yaml_path}')
print(f'클래스 수: {len(names_list)}')
print(f'\n상위 30개:')
for i, name in enumerate(names_list[:30]):
    print(f'  {i:>3}: {name}')

In [ ]:
# Step 6: 검증 — 라벨과 data.yaml 정합성 체크
import random

print('=== 정합성 검증 ===')

for split, paths in SPLITS.items():
    lbl_dir = paths['lbl']
    if not lbl_dir.exists():
        continue
    label_files = [f for f in os.listdir(str(lbl_dir)) if f.endswith('.txt')]
    sample = random.sample(label_files, min(10, len(label_files)))

    print(f'\n[{split}] 랜덤 10개 샘플:')
    for lbl_file in sample:
        stem = lbl_file.rsplit('.', 1)[0]
        menu_from_filename = stem.split('_')[3] if len(stem.split('_')) >= 4 else '?'

        with open(lbl_dir / lbl_file) as f:
            line = f.readline().strip()
        class_id = int(line.split()[0]) if line else -1
        class_name = names_list[class_id] if 0 <= class_id < len(names_list) else '???'

        match = '✅' if menu_from_filename == class_name else '❌'
        print(f'  {match} {lbl_file} → class_id={class_id} → "{class_name}" (파일명: "{menu_from_filename}")')

# class_id 범위 체크
print(f'\ndata.yaml nc={len(names_list)}')
max_id_found = -1
for split, paths in SPLITS.items():
    lbl_dir = paths['lbl']
    if not lbl_dir.exists():
        continue
    for lbl_file in os.listdir(str(lbl_dir)):
        if not lbl_file.endswith('.txt'):
            continue
        with open(lbl_dir / lbl_file) as f:
            for line in f:
                cid = int(line.strip().split()[0])
                max_id_found = max(max_id_found, cid)
print(f'라벨 최대 class_id: {max_id_found}')
print(f'범위 OK: {max_id_found < len(names_list)}')

In [ ]:
# Step 7: 재학습
!pip install -q ultralytics

from ultralytics import YOLO
import torch

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB' if torch.cuda.is_available() else '')

model = YOLO('yolo11s.pt')

results = model.train(
    data=str(DATASET / 'data.yaml'),
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    project=str(Path('/content/drive/MyDrive/runs')),
    name='korean_food_menu_v2',
    exist_ok=True,
    optimizer='auto',
    verbose=True,
)

In [ ]:
# Step 8: 결과 확인
import pandas as pd

results_dir = Path('/content/drive/MyDrive/runs/korean_food_menu_v2')
csv_path = results_dir / 'results.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    last = df.iloc[-1]
    print(f"최종 epoch: {int(last.get('epoch', -1))}")
    print(f"mAP@0.5:    {last.get('metrics/mAP50(B)', 'N/A')}")
    print(f"mAP@0.5:95: {last.get('metrics/mAP50-95(B)', 'N/A')}")
    print(f"Precision:  {last.get('metrics/precision(B)', 'N/A')}")
    print(f"Recall:     {last.get('metrics/recall(B)', 'N/A')}")

best_pt = results_dir / 'weights' / 'best.pt'
if best_pt.exists():
    size_mb = best_pt.stat().st_size / 1e6
    print(f'\nbest.pt: {best_pt} ({size_mb:.1f} MB)')
    print('→ 로컬 Object_Detection/models/best.pt 에 다운로드')
else:
    print('\nbest.pt 아직 생성 안 됨 — 학습 진행 중')

In [ ]:
# Step 9: 메뉴명 매핑 yaml 저장 (로컬에서 사용)
# → Object_Detection/config/food_class_names.yaml 대체용

menu_yaml = {
    'description': f'AI Hub 음식 데이터셋 {len(names_list)}종 메뉴명 매핑 (v2, 메뉴 단위)',
    'names': {i: name for i, name in enumerate(names_list)},
}

out_yaml = DATASET / 'food_menu_names.yaml'
with open(out_yaml, 'w', encoding='utf-8') as f:
    yaml.dump(menu_yaml, f, allow_unicode=True, default_flow_style=False)

print(f'저장: {out_yaml}')
print(f'클래스 수: {len(names_list)}')
print(f'\n→ 학습 완료 후 이 파일을 로컬 Object_Detection/config/food_class_names.yaml 에 복사')
print(f'→ detector.py가 자동으로 메뉴명 라벨 출력')